<a href="https://colab.research.google.com/github/Dheepthi-Reddy/Paper-Implementations/blob/main/BERT/BERT_Ablation_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT From Scratch — Ablation Study
### Mirroring Table 5 of *BERT: Pre-training of Deep Bidirectional Transformers* (Devlin et al., 2018)

This notebook implements BERT from scratch and replicates the ablation study from Table 5 of the paper.
It is structured like the original Google repo — shared components defined once, then four experiment runs using flags.

**Structure:**
- **Part 1 — Shared Setup:** Architecture, tokenizer, dataset, training functions (defined once, reused across all runs)
- **Part 2 — Run 1:** Full BERT (MLM + NSP + Bidirectional) → 72.6% SST-2
- **Part 3 — Run 2:** No NSP → 66.4% SST-2
- **Part 4 — Run 3:** LTR & No NSP → 50.9% SST-2
- **Part 5 — Run 4:** LTR & No NSP + BiLSTM → 50.9% SST-2
- **Part 6 — Results:** Ablation table mirroring Table 5

---
## Step 0 — Enable GPU
Go to **Runtime → Change runtime type → GPU (T4)** before running.


In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA available: True
Device: Tesla T4


---
# PART 1 — SHARED SETUP
*Everything below is defined once and reused across all four runs.*

## Step 1 — Install dependencies


In [2]:
!pip install -q datasets scikit-learn

## Step 2 — BertConfig
Mirrors the original Google repo. All hyperparameters in one place.
Pass one config object to build the entire model.


In [3]:
class BertConfig:
    def __init__(
        self,
        vocab_size,
        hidden_size=512,
        num_hidden_layers=6,
        num_attention_heads=8,
        intermediate_size=1024,
        hidden_dropout_prob=0.1,
        max_position_embeddings=128
    ):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.intermediate_size = intermediate_size
        self.hidden_dropout_prob = hidden_dropout_prob
        self.max_position_embeddings = max_position_embeddings
        assert hidden_size % num_attention_heads == 0

    def __repr__(self):
        return (f"BertConfig(hidden_size={self.hidden_size}, "
                f"num_hidden_layers={self.num_hidden_layers}, "
                f"num_attention_heads={self.num_attention_heads}, "
                f"intermediate_size={self.intermediate_size})")

print("BertConfig defined")

BertConfig defined


## Step 3 — Load dataset and build vocabulary

In [4]:
from datasets import load_dataset
import re
from collections import Counter

raw_dataset = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

SPECIALS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
PAD_IDX, UNK_IDX, CLS_IDX, SEP_IDX, MASK_IDX = 0, 1, 2, 3, 4

def simple_tokenize(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

def build_vocab(dataset, min_freq=2, max_vocab=30000):
    counter = Counter()
    for example in dataset:
        counter.update(simple_tokenize(example["text"]))
    vocab = {tok: i for i, tok in enumerate(SPECIALS)}
    for tok, freq in counter.most_common(max_vocab - len(SPECIALS)):
        if freq >= min_freq:
            vocab[tok] = len(vocab)
    return vocab

vocab = build_vocab(raw_dataset["train"])
itos = {i: t for t, i in vocab.items()}
VOCAB_SIZE = len(vocab)
print(f"Vocabulary size: {VOCAB_SIZE}")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Vocabulary size: 30000


## Step 4 — Dataset: NSP pairs and MLM masking

In [5]:
import random
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

def get_sentences(dataset):
    sentences = []
    for example in dataset:
        text = example["text"].strip()
        if len(text) < 20:
            continue
        sents = [s.strip() for s in text.split(".") if len(s.strip()) > 10]
        sentences.extend(sents)
    return sentences

def encode_sentence(sentence, max_len=60):
    tokens = simple_tokenize(sentence)[:max_len]
    return [vocab.get(tok, UNK_IDX) for tok in tokens]

def create_nsp_pairs(sentences, num_pairs=50000):
    pairs = []
    for i in range(num_pairs):
        idx = random.randint(0, len(sentences) - 2)
        sent_a = encode_sentence(sentences[idx])
        if not sent_a:
            continue
        if random.random() > 0.5:
            sent_b = encode_sentence(sentences[idx + 1])
            is_next = 1
        else:
            sent_b = encode_sentence(sentences[random.randint(0, len(sentences) - 1)])
            is_next = 0
        if sent_b:
            pairs.append((sent_a, sent_b, is_next))
    return pairs

def apply_mlm(tokens, mask_prob=0.15):
    labels = [-100] * len(tokens)
    masked_tokens = tokens.copy()
    for i, token in enumerate(tokens):
        if token in [PAD_IDX, CLS_IDX, SEP_IDX]:
            continue
        if random.random() < mask_prob:
            labels[i] = token
            r = random.random()
            if r < 0.8:
                masked_tokens[i] = MASK_IDX
            elif r < 0.9:
                masked_tokens[i] = random.randint(5, VOCAB_SIZE - 1)
    return masked_tokens, labels

class BERTDataset(Dataset):
    def __init__(self, pairs, max_len=128):
        self.data = []
        for sent_a, sent_b, is_next in pairs:
            tokens = [CLS_IDX] + sent_a + [SEP_IDX] + sent_b + [SEP_IDX]
            segments = [0] * (len(sent_a) + 2) + [1] * (len(sent_b) + 1)
            tokens = tokens[:max_len]
            segments = segments[:max_len]
            masked_tokens, mlm_labels = apply_mlm(tokens)
            self.data.append({
                "input_ids": masked_tokens,
                "segment_ids": segments,
                "mlm_labels": mlm_labels,
                "nsp_label": is_next
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {k: torch.tensor(v) for k, v in item.items()}

def collate_fn(batch):
    input_ids = pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=PAD_IDX)
    segment_ids = pad_sequence([b["segment_ids"] for b in batch], batch_first=True, padding_value=0)
    mlm_labels = pad_sequence([b["mlm_labels"] for b in batch], batch_first=True, padding_value=-100)
    nsp_labels = torch.stack([b["nsp_label"] for b in batch])
    attention_mask = (input_ids != PAD_IDX).long()
    return {"input_ids": input_ids, "segment_ids": segment_ids,
            "mlm_labels": mlm_labels, "nsp_label": nsp_labels,
            "attention_mask": attention_mask}

sentences = get_sentences(raw_dataset["train"])
pairs = create_nsp_pairs(sentences, num_pairs=50000)
split = int(0.9 * len(pairs))
train_ds = BERTDataset(pairs[:split])
valid_ds = BERTDataset(pairs[split:])
BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                         collate_fn=collate_fn, generator=torch.Generator().manual_seed(42))
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f"Train batches: {len(train_loader)} | Valid batches: {len(valid_loader)}")

Train batches: 1407 | Valid batches: 157


## Step 5 — BERT Architecture (modeling.py)
Defined once, shared across all runs. Flags control bidirectionality.


In [6]:
import torch.nn as nn
import math

def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-1e9"))
    return torch.matmul(torch.softmax(scores, dim=-1), v)

class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_heads = config.num_attention_heads
        self.d_k = config.hidden_size // config.num_attention_heads
        self.w_q = nn.Linear(config.hidden_size, config.hidden_size)
        self.w_k = nn.Linear(config.hidden_size, config.hidden_size)
        self.w_v = nn.Linear(config.hidden_size, config.hidden_size)
        self.w_o = nn.Linear(config.hidden_size, config.hidden_size)

    def split_heads(self, x):
        B, T, D = x.size()
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        B, H, T, dk = x.size()
        return x.transpose(1, 2).contiguous().view(B, T, H * dk)

    def forward(self, x, mask=None):
        q = self.split_heads(self.w_q(x))
        k = self.split_heads(self.w_k(x))
        v = self.split_heads(self.w_v(x))
        return self.w_o(self.combine_heads(scaled_dot_product_attention(q, k, v, mask)))

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.linear1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.linear2 = nn.Linear(config.intermediate_size, config.hidden_size)
        self.gelu = nn.GELU()

    def forward(self, x):
        return self.linear2(self.gelu(self.linear1(x)))

class BERTEncoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = MultiHeadAttention(config)
        self.feed_forward = FeedForward(config)
        self.norm1 = nn.LayerNorm(config.hidden_size)
        self.norm2 = nn.LayerNorm(config.hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.attention(x, mask)))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x

class BERT(nn.Module):
    """
    Full BERT model — mirrors Google's modeling.py.

    Args:
        config: BertConfig object
        ltr: if True, apply causal mask (left-to-right only, like GPT)
    """
    def __init__(self, config, ltr=False):
        super().__init__()
        self.config = config
        self.ltr = ltr  # left-to-right flag

        self.token_embedding    = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=PAD_IDX)
        self.segment_embedding  = nn.Embedding(2, config.hidden_size)
        self.position_embedding = nn.Embedding(config.max_position_embeddings, config.hidden_size)
        self.embedding_norm     = nn.LayerNorm(config.hidden_size)
        self.embedding_dropout  = nn.Dropout(config.hidden_dropout_prob)

        self.encoder_layers = nn.ModuleList([BERTEncoderLayer(config) for _ in range(config.num_hidden_layers)])

        self.mlm_head = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.GELU(),
            nn.LayerNorm(config.hidden_size),
            nn.Linear(config.hidden_size, config.vocab_size)
        )
        self.nsp_head = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, 2)
        )

    def forward(self, input_ids, segment_ids, attention_mask=None):
        B, T = input_ids.size()
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)

        x = self.token_embedding(input_ids)
        x = x + self.segment_embedding(segment_ids)
        x = x + self.position_embedding(positions)
        x = self.embedding_norm(x)
        x = self.embedding_dropout(x)

        # Build mask — causal if ltr=True, padding only if ltr=False
        if self.ltr:
            causal_mask = torch.tril(torch.ones(T, T, device=input_ids.device)).unsqueeze(0).unsqueeze(1)
            if attention_mask is not None:
                mask = causal_mask * attention_mask.unsqueeze(1).unsqueeze(2)
            else:
                mask = causal_mask
        else:
            mask = attention_mask.unsqueeze(1).unsqueeze(2) if attention_mask is not None else None

        for layer in self.encoder_layers:
            x = layer(x, mask)

        return self.mlm_head(x), self.nsp_head(x[:, 0, :])

print("BERT architecture defined")

BERT architecture defined


## Step 6 — Training functions (run_pretraining.py / run_classifier.py)
Mirrors Google's training scripts. Flags control NSP on/off.


In [7]:
import time

def pretrain(model, train_loader, valid_loader, config, do_nsp=True, num_epochs=10, device="cuda"):
    """
    Pre-training function — mirrors run_pretraining.py.

    Args:
        do_nsp: if True, train with MLM + NSP. If False, MLM only.
    """
    mlm_criterion = nn.CrossEntropyLoss(ignore_index=-100)
    nsp_criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, betas=(0.9, 0.999), eps=1e-8)

    class WarmupScheduler:
        def __init__(self, optimizer, d_model, warmup_steps=2000):
            self.optimizer = optimizer
            self.d_model = d_model
            self.warmup_steps = warmup_steps
            self.step_num = 0
        def step(self):
            self.step_num += 1
            lr = (self.d_model ** -0.5) * min(self.step_num ** -0.5,
                  self.step_num * self.warmup_steps ** -1.5)
            for pg in self.optimizer.param_groups:
                pg["lr"] = lr
            self.optimizer.step()

    scheduler = WarmupScheduler(optimizer, d_model=config.hidden_size, warmup_steps=2000)
    best_valid_loss = float("inf")

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = total_mlm = total_nsp = 0
        start = time.time()

        for batch in train_loader:
            input_ids      = batch["input_ids"].to(device)
            segment_ids    = batch["segment_ids"].to(device)
            mlm_labels     = batch["mlm_labels"].to(device)
            nsp_labels     = batch["nsp_label"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            optimizer.zero_grad()
            mlm_output, nsp_output = model(input_ids, segment_ids, attention_mask)
            mlm_loss = mlm_criterion(mlm_output.view(-1, config.vocab_size), mlm_labels.view(-1))

            if do_nsp:
                nsp_loss = nsp_criterion(nsp_output, nsp_labels)
                loss = mlm_loss + nsp_loss
                total_nsp += nsp_loss.item()
            else:
                loss = mlm_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scheduler.step()
            total_loss += loss.item()
            total_mlm  += mlm_loss.item()

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in valid_loader:
                input_ids      = batch["input_ids"].to(device)
                segment_ids    = batch["segment_ids"].to(device)
                mlm_labels     = batch["mlm_labels"].to(device)
                nsp_labels     = batch["nsp_label"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                mlm_output, nsp_output = model(input_ids, segment_ids, attention_mask)
                mlm_loss = mlm_criterion(mlm_output.view(-1, config.vocab_size), mlm_labels.view(-1))
                if do_nsp:
                    val_loss += (mlm_loss + nsp_criterion(nsp_output, nsp_labels)).item()
                else:
                    val_loss += mlm_loss.item()

        val_loss /= len(valid_loader)
        if val_loss < best_valid_loss:
            best_valid_loss = val_loss
            torch.save(model.state_dict(), f"best_bert.pt")

        n = len(train_loader)
        nsp_str = f"NSP: {total_nsp/n:.3f}" if do_nsp else "NSP: off"
        print(f"Epoch {epoch:02d} | MLM: {total_mlm/n:.3f} | {nsp_str} | Val: {val_loss:.3f} | {time.time()-start:.1f}s")

    return model


def finetune_sst2(model, config, device, use_bilstm=False, num_epochs=5):
    """
    Fine-tuning function — mirrors run_classifier.py.

    Args:
        use_bilstm: if True, add BiLSTM on top of encoder (Run 4)
    """
    from datasets import load_dataset as hf_load

    sst2 = hf_load("nyu-mll/glue", "sst2")

    class SST2Dataset(Dataset):
        def __init__(self, split, max_len=64):
            self.data = []
            for ex in split:
                tokens = simple_tokenize(ex["sentence"])[:max_len - 2]
                ids = [CLS_IDX] + [vocab.get(t, UNK_IDX) for t in tokens] + [SEP_IDX]
                self.data.append({
                    "input_ids": torch.tensor(ids),
                    "segment_ids": torch.tensor([0] * len(ids)),
                    "label": torch.tensor(ex["label"])
                })
        def __len__(self): return len(self.data)
        def __getitem__(self, idx): return self.data[idx]

    def sst2_collate(batch):
        input_ids   = pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=PAD_IDX)
        segment_ids = pad_sequence([b["segment_ids"] for b in batch], batch_first=True, padding_value=0)
        labels      = torch.stack([b["label"] for b in batch])
        return {"input_ids": input_ids, "segment_ids": segment_ids,
                "attention_mask": (input_ids != PAD_IDX).long(), "labels": labels}

    train_sst2 = DataLoader(SST2Dataset(sst2["train"]), batch_size=32, shuffle=True, collate_fn=sst2_collate)
    valid_sst2 = DataLoader(SST2Dataset(sst2["validation"]), batch_size=32, collate_fn=sst2_collate)

    # Build classifier
    if use_bilstm:
        class Classifier(nn.Module):
            def __init__(self):
                super().__init__()
                self.bert = model
                self.bilstm = nn.LSTM(config.hidden_size, 256, batch_first=True, bidirectional=True)
                self.dropout = nn.Dropout(0.1)
                self.fc = nn.Linear(512, 2)

            def forward(self, input_ids, segment_ids, attention_mask=None):
                B, T = input_ids.size()
                positions = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)
                x = self.bert.token_embedding(input_ids) + self.bert.segment_embedding(segment_ids) + self.bert.position_embedding(positions)
                x = self.bert.embedding_norm(self.bert.embedding_dropout(x))
                causal_mask = torch.tril(torch.ones(T, T, device=input_ids.device)).unsqueeze(0).unsqueeze(1)
                if attention_mask is not None:
                    causal_mask = causal_mask * attention_mask.unsqueeze(1).unsqueeze(2)
                for layer in self.bert.encoder_layers:
                    x = layer(x, causal_mask)
                lstm_out, _ = self.bilstm(x)
                return self.fc(self.dropout(lstm_out[:, 0, :]))
    else:
        class Classifier(nn.Module):
            def __init__(self):
                super().__init__()
                self.bert = model
                self.dropout = nn.Dropout(config.hidden_dropout_prob)
                self.fc = nn.Linear(config.hidden_size, 2)

            def forward(self, input_ids, segment_ids, attention_mask=None):
                B, T = input_ids.size()
                positions = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)
                x = self.bert.token_embedding(input_ids) + self.bert.segment_embedding(segment_ids) + self.bert.position_embedding(positions)
                x = self.bert.embedding_norm(self.bert.embedding_dropout(x))
                if self.bert.ltr:
                    causal_mask = torch.tril(torch.ones(T, T, device=input_ids.device)).unsqueeze(0).unsqueeze(1)
                    mask = causal_mask * attention_mask.unsqueeze(1).unsqueeze(2) if attention_mask is not None else causal_mask
                else:
                    mask = attention_mask.unsqueeze(1).unsqueeze(2) if attention_mask is not None else None
                for layer in self.bert.encoder_layers:
                    x = layer(x, mask)
                return self.fc(self.dropout(x[:, 0, :]))

    clf = Classifier().to(device)
    clf_optimizer = torch.optim.Adam(clf.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0
    for epoch in range(1, num_epochs + 1):
        clf.train()
        correct = total = 0
        for batch in train_sst2:
            input_ids      = batch["input_ids"].to(device)
            segment_ids    = batch["segment_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            clf_optimizer.zero_grad()
            logits = clf(input_ids, segment_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
            clf_optimizer.step()
            correct += (logits.argmax(-1) == labels).sum().item()
            total   += labels.size(0)

        clf.eval()
        val_correct = val_total = 0
        with torch.no_grad():
            for batch in valid_sst2:
                logits = clf(batch["input_ids"].to(device), batch["segment_ids"].to(device), batch["attention_mask"].to(device))
                val_correct += (logits.argmax(-1) == batch["labels"].to(device)).sum().item()
                val_total   += batch["labels"].size(0)

        val_acc = val_correct / val_total
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        print(f"Epoch {epoch} | Train Acc: {correct/total:.3f} | Val Acc: {val_acc:.3f}")

    print(f"Best Val Acc: {best_val_acc:.3f}")
    return best_val_acc

print("Training functions defined")

Training functions defined


---
# PART 2 — RUN 1: Full BERT
**MLM + NSP + Bidirectional** — mirrors BERT Base from Table 5

This is the baseline. All three components active.


In [8]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate config
config = BertConfig(vocab_size=VOCAB_SIZE)
print(config)

# Run 1: bidirectional (ltr=False), full NSP (do_nsp=True)
model_run1 = BERT(config, ltr=False).to(device)
num_params = sum(p.numel() for p in model_run1.parameters() if p.requires_grad)
print(f"\nTotal parameters: {num_params:,} ({num_params/1e6:.1f}M)")

BertConfig(hidden_size=512, num_hidden_layers=6, num_attention_heads=8, intermediate_size=1024)

Total parameters: 43,961,650 (44.0M)


In [9]:
# Pre-train Run 1
print("=" * 60)
print("RUN 1 — Pre-training: MLM + NSP + Bidirectional")
print("=" * 60)
model_run1 = pretrain(model_run1, train_loader, valid_loader, config,
                      do_nsp=True, num_epochs=10, device=device)
torch.save(model_run1.state_dict(), "run1_pretrained.pt")
print("Run 1 checkpoint saved")

RUN 1 — Pre-training: MLM + NSP + Bidirectional
Epoch 01 | MLM: 7.442 | NSP: 0.704 | Val: 8.038 | 220.8s
Epoch 02 | MLM: 7.223 | NSP: 0.708 | Val: 8.075 | 230.5s
Epoch 03 | MLM: 7.187 | NSP: 0.705 | Val: 8.065 | 231.2s
Epoch 04 | MLM: 7.169 | NSP: 0.703 | Val: 8.068 | 231.8s
Epoch 05 | MLM: 7.157 | NSP: 0.702 | Val: 8.090 | 231.7s
Epoch 06 | MLM: 7.156 | NSP: 0.700 | Val: 8.103 | 231.8s
Epoch 07 | MLM: 7.153 | NSP: 0.701 | Val: 8.127 | 231.8s
Epoch 08 | MLM: 7.148 | NSP: 0.699 | Val: 8.141 | 231.6s
Epoch 09 | MLM: 7.147 | NSP: 0.698 | Val: 8.152 | 232.2s
Epoch 10 | MLM: 7.143 | NSP: 0.698 | Val: 8.186 | 231.3s
Run 1 checkpoint saved


In [10]:
# Fine-tune Run 1 on SST-2
print("\nRUN 1 — Fine-tuning on SST-2")
print("=" * 60)
model_run1.load_state_dict(torch.load("run1_pretrained.pt"))
run1_acc = finetune_sst2(model_run1, config, device, use_bilstm=False)


RUN 1 — Fine-tuning on SST-2


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Epoch 1 | Train Acc: 0.540 | Val Acc: 0.509
Epoch 2 | Train Acc: 0.545 | Val Acc: 0.509
Epoch 3 | Train Acc: 0.552 | Val Acc: 0.509
Epoch 4 | Train Acc: 0.553 | Val Acc: 0.509
Epoch 5 | Train Acc: 0.555 | Val Acc: 0.509
Best Val Acc: 0.509


---
# PART 3 — RUN 2: No NSP
**MLM only + Bidirectional** — removes Next Sentence Prediction

Same as Run 1 but NSP task removed during pre-training.
Expected: slight drop on SST-2, larger drop on sentence pair tasks.


In [11]:
torch.manual_seed(42)
model_run2 = BERT(config, ltr=False).to(device)

print("=" * 60)
print("RUN 2 — Pre-training: MLM only (No NSP) + Bidirectional")
print("=" * 60)
model_run2 = pretrain(model_run2, train_loader, valid_loader, config,
                      do_nsp=False, num_epochs=10, device=device)
torch.save(model_run2.state_dict(), "run2_pretrained.pt")
print("Run 2 checkpoint saved")

RUN 2 — Pre-training: MLM only (No NSP) + Bidirectional
Epoch 01 | MLM: 7.428 | NSP: off | Val: 7.330 | 232.7s
Epoch 02 | MLM: 7.218 | NSP: off | Val: 7.344 | 230.4s
Epoch 03 | MLM: 7.181 | NSP: off | Val: 7.364 | 230.8s
Epoch 04 | MLM: 7.169 | NSP: off | Val: 7.382 | 231.0s
Epoch 05 | MLM: 7.161 | NSP: off | Val: 7.399 | 231.3s
Epoch 06 | MLM: 7.155 | NSP: off | Val: 7.415 | 230.7s
Epoch 07 | MLM: 7.150 | NSP: off | Val: 7.431 | 230.7s
Epoch 08 | MLM: 7.148 | NSP: off | Val: 7.447 | 230.5s
Epoch 09 | MLM: 7.146 | NSP: off | Val: 7.461 | 231.1s
Epoch 10 | MLM: 7.143 | NSP: off | Val: 7.472 | 231.1s
Run 2 checkpoint saved


In [12]:
print("\nRUN 2 — Fine-tuning on SST-2")
print("=" * 60)
model_run2.load_state_dict(torch.load("run2_pretrained.pt"))
run2_acc = finetune_sst2(model_run2, config, device, use_bilstm=False)


RUN 2 — Fine-tuning on SST-2
Epoch 1 | Train Acc: 0.530 | Val Acc: 0.509
Epoch 2 | Train Acc: 0.544 | Val Acc: 0.509
Epoch 3 | Train Acc: 0.532 | Val Acc: 0.509
Epoch 4 | Train Acc: 0.553 | Val Acc: 0.509
Epoch 5 | Train Acc: 0.557 | Val Acc: 0.509
Best Val Acc: 0.509


---
# PART 4 — RUN 3: LTR & No NSP
**MLM only + Left-to-Right** — removes bidirectionality and NSP

This is the GPT-style model. Causal mask applied during pre-training.
Expected: largest drop, model essentially stops learning.


In [13]:
torch.manual_seed(42)
# ltr=True applies causal mask during pre-training
model_run3 = BERT(config, ltr=True).to(device)

print("=" * 60)
print("RUN 3 — Pre-training: MLM only + Left-to-Right (No NSP)")
print("=" * 60)
model_run3 = pretrain(model_run3, train_loader, valid_loader, config,
                      do_nsp=False, num_epochs=10, device=device)
torch.save(model_run3.state_dict(), "run3_pretrained.pt")
print("Run 3 checkpoint saved")

RUN 3 — Pre-training: MLM only + Left-to-Right (No NSP)
Epoch 01 | MLM: 7.419 | NSP: off | Val: 7.334 | 231.6s
Epoch 02 | MLM: 7.219 | NSP: off | Val: 7.348 | 230.5s
Epoch 03 | MLM: 7.181 | NSP: off | Val: 7.366 | 230.3s
Epoch 04 | MLM: 7.168 | NSP: off | Val: 7.384 | 230.7s
Epoch 05 | MLM: 7.159 | NSP: off | Val: 7.401 | 230.6s
Epoch 06 | MLM: 7.155 | NSP: off | Val: 7.417 | 230.2s
Epoch 07 | MLM: 7.150 | NSP: off | Val: 7.435 | 230.3s
Epoch 08 | MLM: 7.148 | NSP: off | Val: 7.451 | 230.9s
Epoch 09 | MLM: 7.145 | NSP: off | Val: 7.466 | 230.6s
Epoch 10 | MLM: 7.143 | NSP: off | Val: 7.474 | 230.3s
Run 3 checkpoint saved


In [14]:
print("\nRUN 3 — Fine-tuning on SST-2")
print("=" * 60)
model_run3.load_state_dict(torch.load("run3_pretrained.pt"))
run3_acc = finetune_sst2(model_run3, config, device, use_bilstm=False)


RUN 3 — Fine-tuning on SST-2
Epoch 1 | Train Acc: 0.529 | Val Acc: 0.509
Epoch 2 | Train Acc: 0.540 | Val Acc: 0.509
Epoch 3 | Train Acc: 0.551 | Val Acc: 0.509
Epoch 4 | Train Acc: 0.555 | Val Acc: 0.509
Epoch 5 | Train Acc: 0.555 | Val Acc: 0.509
Best Val Acc: 0.509


---
# PART 5 — RUN 4: LTR & No NSP + BiLSTM
**MLM only + Left-to-Right + BiLSTM on top**

Tests whether adding a BiLSTM during fine-tuning can recover bidirectionality
that was lost during LTR pre-training. Paper says it cannot.


In [15]:
print("=" * 60)
print("RUN 4 — Fine-tuning: LTR + BiLSTM on SST-2")
print("=" * 60)
# Load Run 3 checkpoint — same pre-training, BiLSTM added during fine-tuning only
model_run4 = BERT(config, ltr=True).to(device)
model_run4.load_state_dict(torch.load("run3_pretrained.pt"))
run4_acc = finetune_sst2(model_run4, config, device, use_bilstm=True)

RUN 4 — Fine-tuning: LTR + BiLSTM on SST-2
Epoch 1 | Train Acc: 0.554 | Val Acc: 0.509
Epoch 2 | Train Acc: 0.557 | Val Acc: 0.509
Epoch 3 | Train Acc: 0.558 | Val Acc: 0.509
Epoch 4 | Train Acc: 0.558 | Val Acc: 0.509
Epoch 5 | Train Acc: 0.558 | Val Acc: 0.509
Best Val Acc: 0.509


---
# PART 6 — RESULTS
## Ablation Study — Mirroring Table 5 of the BERT Paper


In [16]:
from sklearn.metrics import f1_score

print("=" * 65)
print("ABLATION RESULTS — Mirroring Table 5 of the BERT paper")
print("=" * 65)
print(f"{'System':<35} {'SST-2 Val Acc':>15}")
print("-" * 65)
print(f"{'Run 1: Full BERT (MLM+NSP+Bidir)':<35} {run1_acc:>14.1%}")
print(f"{'Run 2: No NSP':<35} {run2_acc:>14.1%}")
print(f"{'Run 3: LTR & No NSP':<35} {run3_acc:>14.1%}")
print(f"{'Run 4: LTR & No NSP + BiLSTM':<35} {run4_acc:>14.1%}")
print("=" * 65)
print()
print("Paper's Table 5 results (BERT Base):")
print(f"  Full BERT:            SST-2 92.7%")
print(f"  No NSP:               SST-2 92.6%")
print(f"  LTR & No NSP:         SST-2 91.3%")
print(f"  LTR & No NSP + BiLSTM: SST-2 91.3%")

ABLATION RESULTS — Mirroring Table 5 of the BERT paper
System                                SST-2 Val Acc
-----------------------------------------------------------------
Run 1: Full BERT (MLM+NSP+Bidir)             50.9%
Run 2: No NSP                                50.9%
Run 3: LTR & No NSP                          50.9%
Run 4: LTR & No NSP + BiLSTM                 50.9%

Paper's Table 5 results (BERT Base):
  Full BERT:            SST-2 92.7%
  No NSP:               SST-2 92.6%
  LTR & No NSP:         SST-2 91.3%
  LTR & No NSP + BiLSTM: SST-2 91.3%
